# API End-to-End Test Notebook

Manual testing of the Decision Making Deep Research Assistant API.

**Prerequisites:**
- Backend running at `http://localhost:8000`
- `httpx` installed (`pip install httpx`)

In [1]:
import httpx, json, os

BASE_URL = "http://localhost:8000"

## Health Check

In [2]:
r = httpx.get(f"{BASE_URL}/health")
print(r.json())

{'status': 'ok', 'version': '0.1.0'}


## Autonomous Research (blocking `/invoke`)

In [3]:
query = "What are the main differences between Python and JavaScript for web development?"

r = httpx.post(
    f"{BASE_URL}/research/invoke",
    json={"query": query, "interactive_mode": False},
    timeout=120,
)
result = r.json()
print("Status:", result["status"])
print("Sub-questions:", result.get("sub_questions"))
print("\n--- Final Answer ---\n")
print(result.get("final_answer"))

Status: completed
Sub-questions: ['What are the primary use cases for Python and JavaScript in web development?', 'How do Python and JavaScript differ in terms of runtime environments and execution contexts for web applications?', 'What are the key differences in libraries, frameworks, and ecosystems supporting web development with Python versus JavaScript?', 'How do Python and JavaScript handle asynchronous operations and concurrency in web development?', 'What are the syntactic and structural differences between Python and JavaScript in web development?']

--- Final Answer ---

Python and JavaScript differ significantly in syntax and structure, especially in web development contexts. Here's a structured breakdown of their key differences:

---

### **1. Code Blocks and Syntax**
- **Python**: Uses **indentation** (spaces or tabs) to define code blocks. No braces `{}` are required.
  ```python
  if year == 1991:
      print("Python was released in", year)
  ```
- **JavaScript**: Uses *

## Interactive Mode: Start + Inspect Sub-questions

In [4]:
r = httpx.post(
    f"{BASE_URL}/research/invoke",
    json={"query": "Should I invest in index funds or ETFs?", "interactive_mode": True},
    timeout=60,
)
result = r.json()
conv_id = result["conversation_id"]
print("Status:", result["status"])
print("Sub-questions to review:")
for i, q in enumerate(result.get("sub_questions", []), 1):
    print(f"  {i}. {q}")

Status: awaiting_input
Sub-questions to review:
  1. What are the key differences in investment objectives between index funds and ETFs?
  2. How do the fee structures of index funds and ETFs impact long-term returns?
  3. Which investment vehicle is more tax-efficient for long-term growth?
  4. How does liquidity differ between index funds and ETFs, and why does it matter?
  5. Which option better aligns with my financial goals and risk tolerance?


## Interactive Mode: Approve and Get Final Answer

In [5]:
# Edit sub_questions if desired, then submit
approved = result["sub_questions"]  # or edit the list

r2 = httpx.post(
    f"{BASE_URL}/research/{conv_id}/clarify/invoke",
    json={"sub_questions": approved},
    timeout=360,
)
result2 = r2.json()
print("Status:", result2["status"])
print("\n--- Final Answer ---\n")
print(result2.get("final_answer"))

Status: completed

--- Final Answer ---

To determine which investment option better aligns with your **financial goals** and **risk tolerance**, consider the following structured approach:

---

### **1. Assess Your Risk Tolerance**
- **Definition**: Risk tolerance refers to your psychological and financial capacity to handle market fluctuations. It depends on factors like:
  - **Time Horizon**: Longer horizons (e.g., retirement in 30 years) allow for higher risk, as markets tend to recover over time.
  - **Financial Situation**: Conservative investors nearing retirement or with limited capital may prefer low-risk options.
  - **Personal Comfort**: Even if your financial situation allows for risk, your emotional tolerance for volatility matters. For example, a high-risk portfolio may cause stress during market downturns.

- **Tools to Determine Risk Tolerance**:
  - Use online risk-assessment quizzes (e.g., from financial institutions).
  - Consult a financial advisor for personalized

## Poll Status by Conversation ID

In [6]:
r = httpx.get(f"{BASE_URL}/research/{conv_id}/status")
status_data = r.json()
print("Status:", status_data["status"])
print("Query:", status_data["query"])
print("Final answer present:", bool(status_data.get("final_answer")))

Status: completed
Query: Should I invest in index funds or ETFs?
Final answer present: True


## Finance Query Example

In [7]:
r = httpx.post(
    f"{BASE_URL}/research/invoke",
    json={"query": "What is Apple (AAPL) current stock price and recent financial performance?"},
    timeout=120,
)
result = r.json()
print(result.get("final_answer"))

**Apple (AAPL) Current Stock Price and Recent Financial Performance**  

### **1. Current Stock Price**  
As of the latest available data (likely mid-2025, though some sources reference 2026), Apple’s stock price is **$264.58 USD**, reflecting a **+1.54% increase** from the previous close of **$260.58**. Key metrics include:  
- **Market Cap**: $3.89 trillion  
- **P/E Ratio**: 33.45  
- **Earnings Per Share (EPS)**: $7.91  
- **Dividend Yield**: 39% (note: this appears to be an error; Apple’s actual dividend yield is ~0.5–0.6%, suggesting a data discrepancy).  

---

### **2. Recent Financial Performance (Q3 2024)**  
Apple’s fiscal Q3 2024 results highlight mixed growth and challenges:  
- **Total Revenue**: $94.93 billion (+6.07% YoY), driven by:  
  - **iPhone**: $69.1 billion (-1% YoY)  
  - **Services**: $28.5 billion (+12% YoY), a key growth and stability factor.  
  - **Mac**: $22.5 billion (+14% YoY).  
- **Challenges**: Slowing iPhone growth and regulatory scrutiny in key mar

## SSE Stream (optional, advanced)

Consumes the SSE `/research` endpoint using `httpx` streaming.

In [8]:
with httpx.stream(
    "POST", f"{BASE_URL}/research",
    json={"query": "What is quantum computing?"},
    timeout=180,
) as response:
    for line in response.iter_lines():
        if line.startswith("data:"):
            event = json.loads(line[5:].strip())
            print(f"[{event['type']}]", json.dumps(event, indent=2)[:200])

[start] {
  "type": "start",
  "conversation_id": "fd9bd57a-bfa8-4d0f-aca9-331e6b23bf14",
  "status": "running"
}
[node_start] {
  "type": "node_start",
  "node": "retrieve_memory"
}
[node_end] {
  "type": "node_end",
  "node": "retrieve_memory"
}
[node_start] {
  "type": "node_start",
  "node": "decompose"
}
[sub_questions] {
  "type": "sub_questions",
  "node": "decompose",
  "sub_questions": [
    "What are the fundamental principles of quantum computing and how do they differ from classical computing?",
    "What are 
[node_end] {
  "type": "node_end",
  "node": "decompose"
}
[node_start] {
  "type": "node_start",
  "node": "research"
}
[research_results] {
  "type": "research_results",
  "node": "research",
  "results": [
    {
      "sub_question": "What are the fundamental principles of quantum computing and how do they differ from classical computi
[node_end] {
  "type": "node_end",
  "node": "research"
}
[node_start] {
  "type": "node_start",
  "node": "synthesize"
}
[final_an